In [0]:
import requests
import time
import pandas as pd


def ask_genie(
    question: str,
    space_id: str,
    conversation_id: str = None,
    timeout: int = 120,
    verbose: bool = True,
) -> dict:
    """
    Genie API에 질문을 보내고 결과를 반환하는 함수.

    Args:
        question: 질문 또는 Genie가 되물었을 때의 답변
        space_id: Genie Space ID
        conversation_id: 기존 대화 ID (없으면 새 대화 시작, 있으면 후속 메시지)
        timeout: 폴링 타임아웃(초)
        verbose: 진행 상황 출력 여부

    Returns:
        {
            "conversation_id": str,
            "message_id": str,
            "status": str,           # "COMPLETED" or "FAILED"
            "text": str | None,      # Genie 텍스트 응답
            "sql": str | None,       # 생성된 SQL
            "dataframe": DataFrame | None,  # 쿼리 결과
        }

    사용법:
        # 새 대화
        result = ask_genie("질문", space_id="...")

        # Genie가 되물어보면 → conversation_id 넘겨서 답변
        result = ask_genie("답변", space_id="...", conversation_id=result["conversation_id"])
    """
    host = spark.conf.get("spark.databricks.workspaceUrl")
    token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    base_url = f"https://{host}/api/2.0/genie/spaces/{space_id}"
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    # ========== 1. 메시지 전송 ==========
    if conversation_id is None:
        # 새 대화 시작
        if verbose:
            print(f"새 대화: {question}")
        resp = requests.post(
            f"{base_url}/start-conversation",
            headers=headers,
            json={"content": question},
        )
        resp.raise_for_status()
        data = resp.json()
        conversation_id = data["conversation"]["id"]
        message_id = data["message"]["id"]
    else:
        # 후속 메시지 (답변 또는 추가 질문)
        if verbose:
            print(f"후속 응답: {question}")
        resp = requests.post(
            f"{base_url}/conversations/{conversation_id}/messages",
            headers=headers,
            json={"content": question},
        )
        resp.raise_for_status()
        message_id = resp.json()["id"]

    # ========== 2. 폴링 (COMPLETED 될 때까지 대기) ==========
    poll_url = f"{base_url}/conversations/{conversation_id}/messages/{message_id}"
    start = time.time()
    while time.time() - start < timeout:
        msg = requests.get(poll_url, headers=headers).json()
        status = msg.get("status")
        if verbose:
            print(f"{status}")
        if status in ("COMPLETED", "FAILED"):
            break
        time.sleep(3)
    else:
        raise TimeoutError(f"{timeout}초 내 완료되지 않았습니다.")

    # ========== 3. 결과 파싱 ==========
    result = {
        "conversation_id": conversation_id,
        "message_id": message_id,
        "status": status,
        "text": None,
        "sql": None,
        "dataframe": None,
    }

    if status != "COMPLETED" or not msg.get("attachments"):
        if verbose:
            print(f"{msg.get('error', status)}")
        return result

    for att in msg["attachments"]:
        if att.get("text"):
            result["text"] = att["text"]["content"]
            if verbose:
                print(f"\n{att['text']['content']}")
        if att.get("query"):
            result["sql"] = att["query"]["query"]
            if verbose:
                print(f"\nSQL:\n{att['query']['query']}")
        if att.get("attachment_id"):
            qr_url = f"{base_url}/conversations/{conversation_id}/messages/{message_id}/query-result/{att['attachment_id']}"
            qr = requests.get(qr_url, headers=headers).json()
            if qr.get("columns") and qr.get("data_array"):
                result["dataframe"] = pd.DataFrame(
                    qr["data_array"],
                    columns=[c["name"] for c in qr["columns"]],
                )

    return result

In [0]:
SPACE_ID = "01f148e5845f1f68843892ceb53abd32"

# 1. 새 대화 시작
result = ask_genie("수도권에서 가장 비싼 과일 TOP 5는?", space_id=SPACE_ID)

# 2. Genie가 되물어보면 → conversation_id를 넘겨서 답변
# result = ask_genie(
#     "1kg 기준으로 알려줘",
#     space_id=SPACE_ID,
#     conversation_id=result["conversation_id"],
# )

# 3. 결과 확인
if result["dataframe"] is not None:
    display(result["dataframe"])

In [0]:
SPACE_ID = "01f148e5845f1f68843892ceb53abd32"

# 1. 새 대화 시작
# result = ask_genie("수도권에서 가장 비싼 과일 TOP 5는?", space_id=SPACE_ID)

# 2. Genie가 되물어보면 → conversation_id를 넘겨서 답변
result = ask_genie(
    "1kg 기준으로 알려줘",
    space_id=SPACE_ID,
    conversation_id=result["conversation_id"],
)

# 3. 결과 확인
if result["dataframe"] is not None:
    display(result["dataframe"])